Dataset được lấy từ link https://www.kaggle.com/datasets/psychocoderxd/gpu-performance-prediction
Thực hiện RFECV và PCA với mô hình Ridge regression dự đoán chỉ số GPI của GPU thông qua các chỉ số của phần cứng

| Feature | Data Type | Description |
|---|---|---|
| `cores` | float | Number of CUDA/shading cores |
| `tensor_cores` | float | Number of Tensor cores (AI acceleration) |
| `rt_cores` | float | Number of Ray Tracing cores |
| `sm_count` | float | Streaming Multiprocessors count |
| `base_clock` | float | Base clock frequency (MHz) |
| `boost_clock` | float | Boost clock frequency (MHz) |
| `memory_clock` | float | Memory clock frequency (MHz) |
| `ram` | float | Video memory size (GB) |
| `bus_width` | float | Memory bus width (bits) |
| `fp16_performance` | float | FP16 performance (TFLOPS) |
| `fp32_performance` | float | FP32 performance (TFLOPS) |
| `fp64_performance` | float | FP64 performance (TFLOPS) |
| `bf16_performance` | float | BFloat16 performance (TFLOPS) |
| `tf32_performance` | float | TF32 performance (TFLOPS) |
| `int8_performance` | float | INT8 inference performance (TOPS) |
| `multi_gpu` | float | Multi-GPU support (0/1) |
| `api_cuda` | float | CUDA compute capability version (e.g., 7.5, 8.6) |
| `gpi_value` (target) | float | Graphics Performance Index — higher values indicate better performance |

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/psychocoderxd/gpu-performance-prediction/train.csv
/kaggle/input/datasets/psychocoderxd/gpu-performance-prediction/test.csv


In [2]:
gpu = pd.read_csv('/kaggle/input/datasets/psychocoderxd/gpu-performance-prediction/train.csv')
gpu.head()



,id,cores,tensor_cores,rt_cores,sm_count,base_clock,boost_clock,memory_clock,ram,bus_width,fp16_performance,fp32_performance,fp64_performance,bf16_performance,tf32_performance,int8_performance,multi_gpu,api_cuda,gpi_value
0,1748,6144.0,192.0,48.0,48.0,1575.0,1815.0,1188.0,8.0,256.0,22.300,22.3000,0.3485,NaN,NaN,NaN,NaN,8.6,11.48
1,13263,1920.0,240.0,30.0,30.0,600.0,1215.0,1500.0,6.0,192.0,9.331,4.6660,0.1458,NaN,NaN,NaN,NaN,7.5,3.89
2,1526,4864.0,152.0,38.0,38.0,1410.0,1680.0,1750.0,8.0,256.0,16.340,16.3400,0.2554,NaN,NaN,NaN,NaN,8.6,9.10
3,1862,336.0,NaN,NaN,7.0,NaN,NaN,850.0,1024.0,256.0,NaN,0.8736,0.0728,NaN,NaN,NaN,NaN,2.1,2.84
4,7001,3072.0,384.0,48.0,48.0,1650.0,1860.0,1937.0,8.0,256.0,22.860,11.4300,0.3571,NaN,NaN,NaN,NaN,7.5,7.63


In [3]:
nan_stats = pd.DataFrame({
    'NaN_count': gpu.isna().sum(),
    'NaN_percent': gpu.isna().mean()
})

nan_stats

,NaN_count,NaN_percent
id,0,0.0000
cores,7,0.0007
tensor_cores,6375,0.6375
rt_cores,5779,0.5779
sm_count,3997,0.3997
base_clock,2024,0.2024
boost_clock,2024,0.2024
memory_clock,0,0.0000
ram,0,0.0000
bus_width,0,0.0000


In [4]:
gpu_after_drop_miss = gpu.drop(columns = nan_stats[nan_stats['NaN_percent'] > 0.9].index)
gpu_after_drop_miss.head()

,id,cores,tensor_cores,rt_cores,sm_count,base_clock,boost_clock,memory_clock,ram,bus_width,fp16_performance,fp32_performance,fp64_performance,api_cuda,gpi_value
0,1748,6144.0,192.0,48.0,48.0,1575.0,1815.0,1188.0,8.0,256.0,22.300,22.3000,0.3485,8.6,11.48
1,13263,1920.0,240.0,30.0,30.0,600.0,1215.0,1500.0,6.0,192.0,9.331,4.6660,0.1458,7.5,3.89
2,1526,4864.0,152.0,38.0,38.0,1410.0,1680.0,1750.0,8.0,256.0,16.340,16.3400,0.2554,8.6,9.10
3,1862,336.0,NaN,NaN,7.0,NaN,NaN,850.0,1024.0,256.0,NaN,0.8736,0.0728,2.1,2.84
4,7001,3072.0,384.0,48.0,48.0,1650.0,1860.0,1937.0,8.0,256.0,22.860,11.4300,0.3571,7.5,7.63


In [5]:
nan = pd.DataFrame({
    'NaN_count': gpu_after_drop_miss.isna().sum(),
    'NaN_percent': gpu_after_drop_miss.isna().mean()
})
nan

,NaN_count,NaN_percent
id,0,0.0000
cores,7,0.0007
tensor_cores,6375,0.6375
rt_cores,5779,0.5779
sm_count,3997,0.3997
base_clock,2024,0.2024
boost_clock,2024,0.2024
memory_clock,0,0.0000
ram,0,0.0000
bus_width,0,0.0000


chia train, test , valid

In [6]:
from sklearn.model_selection import train_test_split

gpu_X = gpu_after_drop_miss.drop(columns=['gpi_value', 'id'])
gpu_y = gpu_after_drop_miss['gpi_value']


# 70% test, 15% valid, 15% test

X_train, X_temp, y_train, y_temp = train_test_split(
    gpu_X, gpu_y,
    test_size=0.30,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)


In [7]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
cores,6995.0,3060.021158,3050.788728,8.0000,896.000,2048.0000,4096.0000,21760.0
tensor_cores,2547.0,240.301531,116.157639,24.0000,144.000,240.0000,320.0000,680.0
rt_cores,2952.0,49.783875,23.525771,4.0000,34.000,46.0000,68.0000,170.0
sm_count,4226.0,35.761240,26.422682,1.0000,15.000,30.0000,48.0000,170.0
base_clock,5584.0,1516.645953,446.993945,300.0000,1202.000,1470.0000,1687.7500,2670.0
boost_clock,5584.0,1788.378582,538.186091,601.0000,1392.000,1725.0000,2250.0000,3320.0
memory_clock,7000.0,1540.005143,419.594266,50.0000,1250.000,1600.0000,1752.0000,2518.0
ram,7000.0,121.979571,322.840810,1.0000,4.000,8.0000,12.0000,1792.0
bus_width,7000.0,267.035429,417.948510,32.0000,128.000,256.0000,256.0000,8192.0
fp16_performance,4718.0,20.962841,20.951979,0.0100,5.935,16.2000,29.7700,184.6


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

CleanPipe = Pipeline([('scaler', StandardScaler()), 
                      ('impute', KNNImputer(n_neighbors=5))])

X_train_clean = CleanPipe.fit_transform(X_train)
X_valid_clean = CleanPipe.transform(X_valid)
X_test_clean = CleanPipe.transform(X_test)

**Baseline (Ridge)**

In [9]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_ridge_base = Ridge(alpha=1.0)

model_ridge_base.fit(X_train_clean, y_train)

y_pred_base = model_ridge_base.predict(X_valid_clean)

base_metrics = pd.DataFrame({
    'MAE': [mean_absolute_error(y_valid, y_pred_base)],
    'RMSE': [mean_squared_error(y_valid, y_pred_base) ** 0.5],
    'R2': [r2_score(y_valid, y_pred_base)]
})

print(base_metrics)

        MAE      RMSE        R2
0  0.493542  0.721106  0.987071


**Randomforest (another base)**

In [10]:
from sklearn.ensemble import RandomForestRegressor

model_randomforest = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model_randomforest.fit(X_train_clean, y_train)

y_pred_randomforest = model_randomforest.predict(X_valid_clean)

randomforest_metrics = pd.DataFrame({
    'MAE': [mean_absolute_error(y_valid, y_pred_randomforest)],
    'RMSE': [mean_squared_error(y_valid, y_pred_randomforest) ** 0.5],
    'R2': [r2_score(y_valid, y_pred_randomforest)]
})

print(randomforest_metrics)

        MAE      RMSE       R2
0  0.032833  0.144628  0.99948


**RFECV**

In [11]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

model_ridge_rfecv = Ridge(alpha=1.0)

cv = KFold(
    n_splits=15,
    shuffle=True,
    random_state=42
)

rfecv = RFECV(
    estimator=model_ridge_rfecv,
    step=1,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rfecv.fit(X_train_clean, y_train)

RFECV(cv=KFold(n_splits=15, random_state=42, shuffle=True), estimator=Ridge(),
      n_jobs=-1, scoring='neg_root_mean_squared_error')

In [12]:
print("Optimal number of features:", rfecv.n_features_)
selected_features = X_train.columns[rfecv.support_]  # X_train_clean after KNN -> numpy arr cant use DF
removed_features = X_train.columns[~rfecv.support_]

print(selected_features)
print("removed feature", removed_features)



ranking = pd.Series(
    rfecv.ranking_,
    index=X_train.columns
).sort_values()

print(ranking)

Optimal number of features: 12
Index(['cores', 'tensor_cores', 'sm_count', 'base_clock', 'boost_clock',
       'memory_clock', 'ram', 'bus_width', 'fp16_performance',
       'fp32_performance', 'fp64_performance', 'api_cuda'],
      dtype='object')
removed feature Index(['rt_cores'], dtype='object')
cores               1
tensor_cores        1
sm_count            1
base_clock          1
memory_clock        1
boost_clock         1
ram                 1
bus_width           1
api_cuda            1
fp16_performance    1
fp32_performance    1
fp64_performance    1
rt_cores            2
dtype: int64


In [21]:
corr = X_train.corr()

print(
    corr['rt_cores']
    .sort_values(ascending=False)
)

rt_cores            1.000000
sm_count            0.999538
cores               0.889324
fp32_performance    0.864715
fp16_performance    0.774146
tensor_cores        0.761548
ram                 0.477093
bus_width           0.371744
boost_clock         0.168244
base_clock          0.163215
api_cuda            0.068276
fp64_performance    0.007979
memory_clock       -0.326755
Name: rt_cores, dtype: float64


In [23]:
print(
    corr['base_clock']
    .sort_values(ascending=False)
)

base_clock          1.000000
boost_clock         0.910803
api_cuda            0.757140
fp32_performance    0.620408
cores               0.492167
fp16_performance    0.441134
sm_count            0.332399
memory_clock        0.297267
rt_cores            0.163215
fp64_performance   -0.006756
tensor_cores       -0.055441
bus_width          -0.100224
ram                -0.122514
Name: base_clock, dtype: float64


In [13]:
X_train_rfecv = rfecv.transform(X_train_clean)
X_valid_rfecv = rfecv.transform(X_valid_clean)

model_ridge_rfecv.fit(X_train_rfecv, y_train)
y_pred_ridge_rfecv = model_ridge_rfecv.predict(X_valid_rfecv)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


ridge_rfecv_metrics = pd.DataFrame({
    'MAE': [ mean_absolute_error(y_valid, y_pred_ridge_rfecv)],
    'RMSE': [mean_squared_error(y_valid, y_pred_ridge_rfecv) ** 0.5],
    'R2': [r2_score(y_valid, y_pred_ridge_rfecv)]
})

print(ridge_rfecv_metrics)


        MAE      RMSE       R2
0  0.493281  0.721399  0.98706


**PCA**

In [15]:
from sklearn.decomposition import PCA
#  init
pca = PCA(n_components=0.99)

# PC components
X_train_pca = pca.fit_transform(X_train_clean)

X_valid_pca = pca.transform(X_valid_clean)

X_test_pca = pca.transform(X_test_clean)

In [16]:
print("Original features:", X_train_clean.shape[1])
print("PCA components:", X_train_pca.shape[1])

Original features: 13
PCA components: 10


In [17]:
from sklearn.linear_model import Ridge

model_ridge_pca = Ridge(alpha=1.0)

model_ridge_pca.fit(X_train_pca, y_train)

y_pred_pca = model_ridge_pca.predict(X_valid_pca)

ridge_pca_metrics = pd.DataFrame({
    'MAE': [mean_absolute_error(y_valid, y_pred_pca)],
    'RMSE': [mean_squared_error(y_valid, y_pred_pca) ** 0.5],
    'R2': [r2_score(y_valid, y_pred_pca)]
})

print(ridge_pca_metrics)

        MAE      RMSE        R2
0  0.590348  0.797379  0.984191


In [18]:
print(base_metrics)
print(ridge_rfecv_metrics)
print(ridge_pca_metrics)

        MAE      RMSE        R2
0  0.493542  0.721106  0.987071
        MAE      RMSE       R2
0  0.493281  0.721399  0.98706
        MAE      RMSE        R2
0  0.590348  0.797379  0.984191


In [19]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.cumsum())

[0.51851557 0.17114727 0.08166828 0.07296971 0.05281817 0.04444803
 0.02236632 0.01542568 0.01034775 0.00386487]
[0.51851557 0.68966283 0.77133112 0.84430083 0.897119   0.94156703
 0.96393335 0.97935903 0.98970678 0.99357165]


In [20]:
y_pred_base_test = model_ridge_base.predict(X_test_clean)
base_metrics_test = pd.DataFrame({
    'MAE': [mean_absolute_error(y_test, y_pred_base_test)],
    'RMSE': [mean_squared_error(y_test, y_pred_base_test) ** 0.5],
    'R2': [r2_score(y_test, y_pred_base_test)]
})

X_test_rfecv = rfecv.transform(X_test_clean)

y_pred_ridge_rfecv_test = model_ridge_rfecv.predict(X_test_rfecv)
ridge_rfecv_metrics_test = pd.DataFrame({
    'MAE': [mean_absolute_error(y_test, y_pred_ridge_rfecv_test)],
    'RMSE': [mean_squared_error(y_test, y_pred_ridge_rfecv_test) ** 0.5],
    'R2': [r2_score(y_test, y_pred_ridge_rfecv_test)]
})


X_test_pca = pca.transform(X_test_clean)


y_pred_pca_test = model_ridge_pca.predict(X_test_pca)
ridge_pca_metrics_test = pd.DataFrame({
    'MAE': [mean_absolute_error(y_test, y_pred_pca_test)],
    'RMSE': [mean_squared_error(y_test, y_pred_pca_test) ** 0.5],
    'R2': [r2_score(y_test, y_pred_pca_test)]
})

print(base_metrics_test)
print(ridge_rfecv_metrics_test)
print(ridge_pca_metrics_test)

        MAE      RMSE       R2
0  0.495084  0.745658  0.98658
       MAE      RMSE        R2
0  0.49444  0.745586  0.986582
        MAE      RMSE        R2
0  0.596292  0.825238  0.983562
